In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = 'https://github.com/hajousfn-web/nfl-combai-edge-2027.git'
REPOSITORY_NAME = 'nfl-combai-edge-2027'
working_directory = Path.cwd().resolve()
search_roots = (working_directory, *working_directory.parents)
project_root = next(
    (candidate for candidate in search_roots if (candidate / 'src' / 'pipeline.py').is_file()),
    None,
)

if project_root is None:
    clone_directory = working_directory / REPOSITORY_NAME
    if clone_directory.exists():
        if (clone_directory / 'src' / 'pipeline.py').is_file():
            project_root = clone_directory
        else:
            raise RuntimeError(
                f'Cannot use existing clone destination without src/pipeline.py: {clone_directory}'
            )
    else:
        try:
            subprocess.run(
                ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(clone_directory)],
                check=True,
                capture_output=True,
                text=True,
            )
        except FileNotFoundError as error:
            raise RuntimeError('Git is required to clone the project repository.') from error
        except subprocess.CalledProcessError as error:
            details = error.stderr.strip() or error.stdout.strip() or str(error)
            raise RuntimeError(f'Could not clone the project repository: {details}') from error
        project_root = clone_directory

if not (project_root / 'src' / 'pipeline.py').is_file():
    raise FileNotFoundError(f'Project source tree is missing: {project_root / "src"}')

os.chdir(project_root)
print(f'Project repository ready at: {Path.cwd()}')

# VisiShield-Edge: NFL Big Data Bowl 2027

**Offline 10 Hz tracking + nested/forward Ridge validation + statistical audit + edge FSM telemetry.**

The innovation is a dual-layer architecture: raw spatial tracking becomes auditable 10 Hz kinematics and Cut Efficiency (CE) for offline EPA/YAC evaluation, while an independent deployment-owned policy feeds an offline reference FSM and telemetry ring. Nested Ridge CV, forward draft-class validation, and partial Spearman/permutation analysis are offline statistical evidence only; they neither produce safety classifications nor define thresholds. The software state contract includes SAFE, WARNING, DANGER, EMERGENCY_STOP, SENSOR_FAULT, and FAIL_SAFE_LOCKED with explicit reset acknowledgement. It has no hardware drivers, no certified controller, and no hard-real-time or sub-10 ms guarantee. No results are claimed from mock inputs.

## 1. Environment and bounded local inputs

Locate and import the project modules and list expected local inputs. No raw data are read in this setup cell. Keep tracking input at or below the pipeline's 250,000-row default limit, and use complete player tracks rather than truncating a larger file.

In [ ]:
from pathlib import Path
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'src' / 'pipeline.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the repository root containing src/pipeline.py. '
        'Open this notebook from the project checkout or configure PROJECT_ROOT.'
    )

SRC = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
TRACKING_CSV = DATA_DIR / 'combine_tracking.csv'
PERFORMANCE_CSV = DATA_DIR / 'player_season_outcomes.csv'
sys.path.insert(0, str(SRC))

import edge_safety_fsm
import feature_engineering
import model_integration
import pipeline
import execution_ring
import safety_engine
import safety_kernel
import statistical_audit
import visualization
import zone_loader

assert callable(feature_engineering.run_feature_pipeline)
assert callable(model_integration.run_model_pipeline)
assert callable(pipeline.run_full_pipeline)
assert callable(statistical_audit.run_statistical_audit)
assert callable(visualization.generate_project_visualizations)
assert callable(visualization.generate_audit_visualizations)
assert callable(edge_safety_fsm.EdgeSafetyFSM)
assert callable(safety_kernel.SafetyKernel)
assert callable(safety_engine.SafetyEngine)
assert callable(execution_ring.DeterministicExecutionRing)
assert callable(zone_loader.load_zone_thresholds)

print(f'Project root: {PROJECT_ROOT}')
print('Imported analysis, visualization, FSM, zone, ring, kernel, and offline safety-engine modules.')
print(f'Tracking input present: {TRACKING_CSV.is_file()}')
print(f'Performance input present: {PERFORMANCE_CSV.is_file()}')
print('No input data have been loaded.')

## 2. Offline pipeline and module self-validation

Run all ten modules' synthetic-data tests. These exercise feature extraction, nested Ridge workflow, orchestration, linkage/statistical audits, visualization, FSM lockout behavior, zone configuration validation, proximity hysteresis, sensor-fault latching, and bounded-ring behavior without competition files.

In [ ]:
self_test_modules = [
    'feature_engineering.py',
    'model_integration.py',
    'pipeline.py',
    'statistical_audit.py',
    'edge_safety_fsm.py',
    'zone_loader.py',
    'execution_ring.py',
    'safety_kernel.py',
    'safety_engine.py',
    'visualization.py',
]
for module_name in self_test_modules:
    test = subprocess.run(
        [sys.executable, str(SRC / module_name), '--self-test'],
        cwd=PROJECT_ROOT,
        check=False,
        text=True,
        capture_output=True,
    )
    print(f'--- {module_name} ---')
    print(test.stdout)
    if test.stderr:
        print(test.stderr)
    if test.returncode != 0:
        raise RuntimeError(f'{module_name} self-test failed.')
print(f'All {len(self_test_modules)} module self-tests passed.')

### Offline LiDAR/kinematics ring demonstration

The following uses synthetic distances and an explicitly illustrative threshold profile (including 1.1 m), not venue-approved limits or physical LiDAR. The fixed-capacity ring, hysteresis evaluator, six-state telemetry contract, and manual reset are software-only reference components. The timing output reports this process's observed latency distribution; it is not a <10 ms guarantee and does not bypass Python garbage collection.

In [ ]:
from zone_loader import ZoneThresholds

mock_thresholds = ZoneThresholds(
    warning_enter_m=1.1,
    warning_clear_m=1.2,
    danger_enter_m=0.8,
    danger_clear_m=0.9,
    emergency_enter_m=0.4,
    emergency_clear_m=0.5,
)
engine = safety_engine.SafetyEngine(
    thresholds=mock_thresholds,
    ring_capacity=8,
    recovery_samples_required=1,
    reset_safe_samples_required=2,
    zone_id='illustrative-only',
)
base_timestamp_ns = 1_798_000_000_000_000_000
demo_readings = [0.3, 1.3, 1.3]
for index, distance_m in enumerate(demo_readings):
    engine.submit_reading(
        timestamp_ns=base_timestamp_ns + index * 100_000_000,
        distance_m=distance_m,
        speed_mps=2.0,
        acceleration_mps2=0.1,
        cut_efficiency=0.9,
    )
    event = engine.process_next()
    print(event.as_dict())
reset = engine.manual_reset(
    timestamp_ns=base_timestamp_ns + 300_000_000,
    operator_id='mock-reviewer',
    safe_to_reset=True,
)
print('Manual software reset:', reset.as_dict())
print('Observed mock-loop latency statistics (microseconds):', safety_engine.benchmark_processing(iterations=100, thresholds=mock_thresholds))

## 3. Offline feature engineering and nested/forward Ridge evaluation

When the bounded tracking CSV exists, run feature extraction, nested Ridge evaluation, partial Spearman/permutation audits, and forward validation by draft class when outcomes are supplied. The pipeline validates required linkage columns, rejects duplicate frame keys and over-limit inputs, and filters non-PLAYER entities when labels are available. An optional `launch_frame_id` enables a prelaunch speed/acceleration audit; without a verified event marker, no claim about starting from motion or rest is possible. Rookie filtering uses `draft_year` and `season == draft_year`; check source schema and cohort before interpreting results.

In [ ]:
if TRACKING_CSV.is_file():
    from pipeline import run_full_pipeline

    performance_input = PERFORMANCE_CSV if PERFORMANCE_CSV.is_file() else None
    result = run_full_pipeline(
        tracking_data=TRACKING_CSV,
        performance_data=performance_input,
    )
    print(f'Player summaries: {len(result.player_summary):,} rows')
    if result.model_metrics is None:
        print('No performance outcomes supplied; model evaluation was skipped.')
    else:
        display(result.model_metrics)
        if result.partial_correlations is not None:
            display(result.partial_correlations)
        if result.forward_validation is not None:
            display(result.forward_validation)
    if result.prelaunch_audit is not None:
        display(result.prelaunch_audit)
    else:
        print('No launch_frame_id supplied; pre-motion status is unverified.')
else:
    print('Skipped: add an authorized, bounded tracking CSV under data/.')

## 4. Visualization validation and available artifacts

The visualization self-test uses mock inputs and removes temporary test artifacts. Production figures are written only to this repository's `outputs/` directory. Actual-vs-predicted plots require a separate paired out-of-sample prediction table; aggregate cross-validation metrics alone are not sufficient. Audit charts use verified CSV outputs when available and otherwise render explicit no-data status; the FSM transition figure is a clearly labeled contract illustration, not an empirical risk result.

In [ ]:
visualization_test = subprocess.run(
    [sys.executable, str(SRC / 'visualization.py'), '--self-test'],
    cwd=PROJECT_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(visualization_test.stdout)
if visualization_test.stderr:
    print(visualization_test.stderr)
if visualization_test.returncode != 0:
    raise RuntimeError('Visualization self-test failed.')

plot_tracking = OUTPUTS_DIR / 'tracking_features.csv'
plot_predictions = OUTPUTS_DIR / 'model_predictions.csv'
if plot_tracking.is_file() or plot_predictions.is_file():
    from visualization import generate_project_visualizations

    plots = generate_project_visualizations(
        tracking_data=plot_tracking if plot_tracking.is_file() else None,
        predictions_data=plot_predictions if plot_predictions.is_file() else None,
    )
    for plot_name, plot_path in plots.items():
        print(f'{plot_name}: {plot_path}')
else:
    print('No prepared tracking/prediction CSVs found; only audit status and FSM figures will be generated.')

audit_plots = visualization.generate_audit_visualizations()
for plot_name, plot_path in audit_plots.items():
    print(f'{plot_name}: {plot_path}')
print('Visualization self-test passed.')

## 5. Edge risk evaluation and FSM telemetry contract

The deployment-owned policy classifies each current 10 Hz kinematics/CE observation as `SAFE`, `WARNING`, `DANGER`, or `EMERGENCY_STOP`; approved thresholds are intentionally not asserted. The software FSM also distinguishes `SENSOR_FAULT` and `FAIL_SAFE_LOCKED`. It applies immediate escalation, hysteresis/debouncing, ordered telemetry, and explicit manual reset acknowledgement after consecutive safe samples. This demonstration supplies synthetic classifications to validate state handling only. It is not a measured NFL risk evaluation, certified controller, physical LOTO procedure, or actuator interface.

In [ ]:
from datetime import datetime, timedelta, timezone

fsm = edge_safety_fsm.EdgeSafetyFSM(
    recovery_samples_required=2,
    reset_safe_samples_required=2,
)
start = datetime.now(timezone.utc)
mock_classifications = [
    ('SAFE', {'speed': 2.1, 'proprietary_cut_efficiency': 0.91}),
    ('WARNING', {'speed': 3.0, 'proprietary_cut_efficiency': 0.72}),
    ('DANGER', {'speed': 4.0, 'proprietary_cut_efficiency': 0.48}),
    ('EMERGENCY_STOP', {'speed': 4.5, 'proprietary_cut_efficiency': 0.31}),
    ('SAFE', {'speed': 1.2, 'proprietary_cut_efficiency': 0.95}),
    ('SAFE', {'speed': 1.1, 'proprietary_cut_efficiency': 0.96}),
]
fsm_events = [
    fsm.observe(
        timestamp=start + timedelta(milliseconds=100 * index),
        risk_state=classification,
        metrics=metrics,
    )
    for index, (classification, metrics) in enumerate(mock_classifications)
]
reset_event = fsm.manual_reset(
    timestamp=start + timedelta(milliseconds=700),
    operator_id='mock-reviewer',
    safe_to_reset=True,
)
assert reset_event.state is edge_safety_fsm.SafetyState.SAFE
assert not reset_event.lockout_active
display([event.as_dict() for event in (*fsm_events, reset_event)])

## 6. Interpretation, limitations, and deployment gates

- Treat correlations and cross-validation scores as descriptive evidence, not causal claims.
- Confirm player identifiers, measurement units, table grain, missing-value conventions, and the rookie-season join against the authorized competition schema.
- Do not interpret synthetic self-test output as competition results.
- The current model pipeline saves aggregate evaluation metrics; it does not persist per-player predictions. Build and review an out-of-fold prediction table before using actual-vs-predicted charts.
- Nested Ridge CV is offline statistical evaluation; it does not generate the edge risk classification and must not be used as a safety controller.
- The FSM requires deployment-approved risk thresholds and independently validated invalid/stale-data handling. Its manual reset is a telemetry acknowledgement, not physical lockout/tagout or safety certification.
- Partial-rank permutation inference assumes independent/exchangeable rows; review clustering and event dependence before interpreting p-values. Forward validation must use earlier draft classes only and report any insufficient-training folds.
- A pre-motion claim requires an event-aligned `launch_frame_id` and sufficient prelaunch samples. No launch marker or real data is present by default, so this notebook makes no claim that the test began in motion or from rest.
- If telemetry or figures are selected for persistence, export them under the repository `outputs/` directory; do not write generated assets beside raw inputs or assume mock values are deployment thresholds.
- Keep raw and generated data local, and have a human reviewer approve cohort decisions and strategic conclusions.